In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("/content/netflix_titles.csv")

In [3]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [4]:
df.columns

Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description'],
      dtype='object')

In [5]:
df.shape

(8807, 12)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


In [7]:
df.isnull().sum()

,0
show_id,0
type,0
title,0
director,2634
cast,825
country,831
date_added,10
release_year,0
rating,4
duration,3


In [8]:
df['director'] = df['director'].fillna('Unknown')


In [9]:
df['cast'] = df['cast'].fillna('Not Available')


In [10]:
df['country'] = df['country'].fillna(df['country'].mode()[0])


In [11]:
df = df.dropna(subset=['date_added'])


In [12]:
df['rating'] = df['rating'].fillna(df['rating'].mode()[0])


In [13]:
movie_duration = df[df['type']=='Movie']['duration'].mode()[0]
tv_duration = df[df['type']=='TV Show']['duration'].mode()[0]

df.loc[(df['duration'].isnull()) & (df['type']=='Movie'), 'duration'] = movie_duration
df.loc[(df['duration'].isnull()) & (df['type']=='TV Show'), 'duration'] = tv_duration


In [14]:
df.isnull().sum()


,0
show_id,0
type,0
title,0
director,0
cast,0
country,0
date_added,0
release_year,0
rating,0
duration,0


In [15]:
df.duplicated().sum()

np.int64(0)

In [16]:
df.describe()

,release_year
count,8797.000000
mean,2014.183472
std,8.822191
min,1925.000000
25%,2013.000000
50%,2017.000000
75%,2019.000000
max,2021.000000


In [17]:
df['date_added'] = pd.to_datetime(df['date_added'], format='mixed')

In [18]:
#Extract new features:
df['added_year'] = df['date_added'].dt.year
df['added_month'] = df['date_added'].dt.month_name()

In [19]:
df[['duration_num','duration_type']] = df['duration'].str.split(' ', expand=True)
df['duration_num'] = df['duration_num'].astype(int)

In [22]:
#STEP 7: Duration Analysis
df[df['type']=='Movie']['duration_num'].mean()

np.float64(103.90742606531144)

In [23]:
#Average seasons:
df[df['type']=='TV Show']['duration_num'].mean()

np.float64(1.7461011514356508)

In [24]:
# Remove extra spaces
df['listed_in'] = df['listed_in'].str.strip()

# Split multiple genres                                     #<-----this for genre my mentor

df['listed_in'] = df['listed_in'].str.split(', ')

# Explode into separate rows
df = df.explode('listed_in')

In [25]:
df['country'] = df['country'].str.strip()
df['country'] = df['country'].str.split(', ')
df = df.explode('country')   #for a country normalize

In [26]:
df['rating'].unique()

array(['PG-13', 'TV-MA', 'PG', 'TV-14', 'TV-PG', 'TV-Y', 'TV-Y7', 'R',
       'TV-G', 'G', 'NC-17', '74 min', '84 min', '66 min', 'NR',
       'TV-Y7-FV', 'UR'], dtype=object)

In [27]:
df['rating'] = df['rating'].str.upper().str.strip()

In [28]:
df['type'] = df['type'].str.strip().str.title()

In [33]:
categorical_cols = ['type','rating','listed_in','country']

for col in categorical_cols:
    df[col] = df[col].astype('category')

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 23734 entries, 0 to 8806
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   show_id        23734 non-null  object        
 1   type           23734 non-null  category      
 2   title          23734 non-null  object        
 3   director       23734 non-null  object        
 4   cast           23734 non-null  object        
 5   country        23734 non-null  category      
 6   date_added     23734 non-null  datetime64[ns]
 7   release_year   23734 non-null  int64         
 8   rating         23734 non-null  category      
 9   duration       23734 non-null  object        
 10  listed_in      23734 non-null  category      
 11  description    23734 non-null  object        
 12  added_year     23734 non-null  int32         
 13  added_month    23734 non-null  object        
 14  duration_num   23734 non-null  int64         
 15  duration_type  23734 non-